In [2]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download resources once
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

# Load text dataset
file_path = r"C:\Users\ujjwa\Downloads\complaint-intelligence-platform\data\processed\complaints_text.csv"
df = pd.read_csv(file_path, low_memory=False)

print("Shape:", df.shape)
print(df[["complaint_text"]].head())

# NLP tools
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = str(text).lower()

    # remove masked tokens like xxxx
    text = re.sub(r"\bxx+\b", " ", text)

    # remove numbers and punctuation
    text = re.sub(r"[^a-z\s]", " ", text)

    # remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    tokens = text.split()

    # remove stopwords and short words
    tokens = [word for word in tokens if word not in stop_words and len(word) > 2]

    # lemmatize words
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return " ".join(tokens)

# Apply cleaning
df["clean_text"] = df["complaint_text"].apply(clean_text)

# Keep only rows with usable text
df = df[df["clean_text"].str.strip() != ""]

print("\nCleaned text sample:")
print(df[["complaint_text", "clean_text"]].head())

# Save cleaned text file
output_file = r"C:\Users\ujjwa\Downloads\complaint-intelligence-platform\data\processed\complaints_text_clean.csv"
df.to_csv(output_file, index=False)

print("\nSaved cleaned NLP file to:")
print(output_file)
print("Final shape:", df.shape)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ujjwa\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ujjwa\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\ujjwa\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Shape: (30000, 12)
                                      complaint_text
0  Issue summarized : I've been unable to buy/sel...
1  I sent my firt letter XX/XX/1922 to investigat...
2  On XX/XX/XXXXXXXX  and XX/XX/XXXX Letters were...
3  I have reached out to the business several tim...
4  Could you please take a moment to review and e...

Cleaned text sample:
                                      complaint_text  \
0  Issue summarized : I've been unable to buy/sel...   
1  I sent my firt letter XX/XX/1922 to investigat...   
2  On XX/XX/XXXXXXXX  and XX/XX/XXXX Letters were...   
3  I have reached out to the business several tim...   
4  Could you please take a moment to review and e...   

                                          clean_text  
0  issue summarized unable buy sell crypto coin w...  
1  sent firt letter investigate following please ...  
2  letter sent credit reporting agency agency sta...  
3  reached business several time via phone via wr...  
4  could please take moment r

# topic modelling

In [3]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

file_path = r"C:\Users\ujjwa\Downloads\complaint-intelligence-platform\data\processed\complaints_text_clean.csv"
df = pd.read_csv(file_path, low_memory=False)

# Use a sample if you want faster execution
df_sample = df.sample(min(20000, len(df)), random_state=42).copy()

# Convert text into TF-IDF features
vectorizer = TfidfVectorizer(
    max_features=2000,
    min_df=10,
    max_df=0.90,
    ngram_range=(1, 2)
)

X = vectorizer.fit_transform(df_sample["clean_text"])

# Run NMF topic model
n_topics = 8
nmf_model = NMF(n_components=n_topics, random_state=42)
W = nmf_model.fit_transform(X)
H = nmf_model.components_

feature_names = vectorizer.get_feature_names_out()

# Print top words per topic
for topic_idx, topic in enumerate(H):
    top_words = [feature_names[i] for i in topic.argsort()[:-11:-1]]
    print(f"Topic {topic_idx + 1}: {', '.join(top_words)}")

Topic 1: credit, report, credit report, account, information, reporting, bureau, inaccurate, credit bureau, inquiry
Topic 2: section, state, right, reporting, section state, violated right, agency furnish, furnish account, state right, right privacy
Topic 3: cash app, app, cash, unfair, resolution, financial, act, unprotected, fraud platform, service unfair
Topic 4: account, bank, payment, card, told, would, money, called, time, back
Topic 5: debt, collection, validation, original, alleged, debt collection, company, collect, alleged debt, provide
Topic 6: consumer, information, consumer reporting, reporting agency, item information, agency, person, reporting, consumer report, usc
Topic 7: zelle, concern, cfpb, transaction, recent cfpb, issue platform, cfpb lawsuit, offender, express concern, light recent
Topic 8: theft, identity, identity theft, balance, victim, victim identity, balance balance, item, block, information
